# DataAnalysisAgent - 智能数据分析助手

## 第1部分：环境配置

In [1]:
# 导入库和配置
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools import Tool, ToolParameter
from typing import Dict, Any, List
import ast
import os

# 配置LLM参数
os.environ["LLM_MODEL_ID"] = "Qwen/Qwen3-8B"
os.environ["LLM_API_KEY"] = "ms-9382e20f-96c2-456a-b609-af5c81201066"
os.environ["LLM_BASE_URL"] = "https://api-inference.modelscope.cn/v1/"
# 增大超时时间，避免数据量较大时请求超时
os.environ["LLM_TIMEOUT"] = "120"

print("✅ 库导入和配置完成")

✅ 库导入和配置完成


## 第2部分：定义数据分析工具

In [2]:
import json
import pandas as pd
from typing import Dict, Any, List

class DataCleaningTool(Tool):
    """数据清洗工具 - 基于用户指定规则清洗表格数据"""

    def __init__(self):
        super().__init__(
            name="data_cleaner",
            description="对传入的表格数据执行清洗操作，包括去空值、列筛选等"
        )

    def run(self, parameters: Dict[str, Any]) -> str:
        """
        执行数据清洗
        parameters 应包含：
          - data_json: str，来自 excel_reader 的 JSON 字符串（必须）
          - drop_na: bool，是否删除含空值的行（默认 True，改为默认开启更实用）
          - columns_to_keep: List[str]，保留的列名列表（可选）
        """
        data_json = parameters.get("data_json")
        if not data_json:
            return "错误：缺少原始数据（data_json 不能为空）"

        try:
            # 解析原始数据
            raw_data = json.loads(data_json)
            records = raw_data.get("完整数据", [])
            if not records:
                return "警告：原始数据为空，无法清洗"

            df = pd.DataFrame(records)

            # 1. 列筛选
            columns_to_keep = parameters.get("columns_to_keep")
            if columns_to_keep:
                missing_cols = [col for col in columns_to_keep if col not in df.columns]
                if missing_cols:
                    return f"错误：指定保留的列不存在：{missing_cols}"
                df = df[columns_to_keep]

            # 2. 去空值（默认开启，实际分析场景中空行通常无意义）
            drop_na = parameters.get("drop_na", True)
            if drop_na:
                df = df.dropna()


✅ DataCleaningTool 定义完成
